# Notebook 11 — U-Net Semantic Segmentation

## Learning Objectives
- Understand semantic segmentation: assign a class label to every pixel.
- Learn the U-Net encoder-decoder architecture with skip connections.
- Train `UNet` on synthetic shapes (background / circle / square / triangle).
- Visualise input images, ground-truth masks, and predicted masks side-by-side.
- Evaluate with pixel accuracy, mean IoU, and Dice score.
- Identify failure cases: class imbalance, boundary errors, small object misdetection.

## Big Picture
Classification assigns one label to the whole image. Segmentation assigns a label to each pixel.
U-Net solves this by encoding the image into a compact representation and then decoding it back
to the original spatial resolution, using *skip connections* to recover fine-grained spatial detail
that is lost during downsampling.

## Dataset
**SyntheticShapes** — procedurally generated 128×128 RGB images with 1–3 geometric shapes each.
4 classes: 0=background, 1=circle (red), 2=square (blue), 3=triangle (green).

## Architecture
```
Input [B, 3, 128, 128]
  ┌─ Encoder ──────────────────────────────────────────────────────────┐
  │  DownBlock1: DoubleConv(3→32)   + MaxPool  → skip1:[B,32,128,128] x:[B,32,64,64]
  │  DownBlock2: DoubleConv(32→64)  + MaxPool  → skip2:[B,64,64,64]   x:[B,64,32,32]
  │  DownBlock3: DoubleConv(64→128) + MaxPool  → skip3:[B,128,32,32]  x:[B,128,16,16]
  └──────────────────────────────────────────────────────────────────────
  Bottleneck: DoubleConv(128→256)              → [B, 256, 16, 16]
  ┌─ Decoder (with skip connections) ─────────────────────────────────┐
  │  UpBlock3: Upsample + cat(skip3) → DoubleConv(384→128) [B,128,32,32]
  │  UpBlock2: Upsample + cat(skip2) → DoubleConv(192→64)  [B,64,64,64]
  │  UpBlock1: Upsample + cat(skip1) → DoubleConv(96→32)   [B,32,128,128]
  └──────────────────────────────────────────────────────────────────────
  Output: Conv1x1(32→4) → logits [B, 4, 128, 128]
```

## Theory
**Semantic segmentation** = pixel-wise classification. The model must produce a label for every
single pixel in the image.

**Skip connections** are the key innovation of U-Net:
- The encoder downsamples, building rich semantic features but losing spatial precision.
- Skip connections pass the high-resolution feature maps from the encoder directly to the decoder.
- The decoder upsamples and combines these features to recover fine spatial detail.

**DoubleConv** block: two consecutive Conv→BN→ReLU layers. Cheaper than a single larger kernel.

**Loss**: cross-entropy over all pixels:
$$\mathcal{L} = -\frac{1}{H \cdot W} \sum_{h,w} \sum_c y_{h,w,c} \log(\hat{p}_{h,w,c})$$

**Metrics**:
- *Pixel accuracy* = fraction of correctly classified pixels.
- *Mean IoU* = average per-class intersection-over-union (ignores absent classes).
- *Dice score* = $2|P \cap G| / (|P| + |G|)$ — commonly used in medical imaging.

**Reference**: Ronneberger, Fischer & Brox, *U-Net: Convolutional Networks for Biomedical
Image Segmentation*, MICCAI 2015.

## Implementation from Scratch
### 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root when running from notebooks/

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, update_runs_summary
from src.utils.paths import RUNS_SEGMENTATION_DIR
from src.data.synthetic_shapes import load_shapes_data, SyntheticShapesDataset
from src.models.unet import UNet
from src.metrics.segmentation import pixel_accuracy, mean_iou, dice_score
from src.visualization.segmentation import overlay_segmentation_mask, plot_segmentation_examples

seed_everything(42)
device = get_best_device()
run_dir = prepare_run_dir(RUNS_SEGMENTATION_DIR, clean=False)
print(f'Device  : {device}')
print(f'Run dir : {run_dir}')

## Dataset
### 2. Load Synthetic Shapes (segmentation mode)

In [ ]:
IMAGE_SIZE    = 128
NUM_CLASSES   = 4   # 0=background, 1=circle, 2=square, 3=triangle
BATCH_SIZE    = 16
NUM_EPOCHS    = 5
LR            = 1e-3
BASE_CHANNELS = 32
CLASS_NAMES   = ['background', 'circle', 'square', 'triangle']

train_loader, val_loader = load_shapes_data(
    n_train=800,
    n_val=200,
    image_size=IMAGE_SIZE,
    mode='segmentation',
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

imgs, masks = next(iter(train_loader))
print(f'Image batch shape : {imgs.shape}   (B, C, H, W)')
print(f'Mask  batch shape : {masks.shape}  (B, H, W) — integer class indices')
print(f'Mask unique values: {masks.unique().tolist()}  (0=background, 1=circle, 2=square, 3=triangle)')
print(f'Pixel range       : [{imgs.min():.3f}, {imgs.max():.3f}]')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

In [ ]:
# Visualise 3 sample images with their ground-truth masks
imgs_np  = imgs[:3].numpy()   # [3, 3, 128, 128]
masks_np = masks[:3].numpy()  # [3, 128, 128]

overlay_segmentation_mask(
    image=imgs_np[0],
    mask=masks_np[0],
    alpha=0.5,
    save_path=run_dir / 'unet_sample_overlay.png',
    title='Sample Image + Ground-Truth Segmentation Mask',
    class_names=CLASS_NAMES,
)
print('Sample overlay saved.')

### 3. Build U-Net

In [ ]:
model = UNet(
    in_channels=3,
    num_classes=NUM_CLASSES,
    base_channels=BASE_CHANNELS,
).to(device)

n_params = model.count_parameters()
print(f'Trainable parameters: {n_params:,}')

# Shape trace
x_sample = imgs[:2].to(device)         # [2, 3, 128, 128]
with torch.no_grad():
    logits = model(x_sample)            # [2, 4, 128, 128]

print(f'Input  shape : {x_sample.shape}   (batch, 3, H, W)')
print(f'Output shape : {logits.shape}  (batch, num_classes, H, W)')
print('Same spatial resolution as input — check!')

## Sanity Checks

In [ ]:
# 1. Output shape matches input spatial size
assert logits.shape == (2, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), f'Shape mismatch: {logits.shape}'
print('Output shape check passed.')

# 2. No NaN
assert not torch.isnan(logits).any()
print('No NaN in output.')

# 3. Random predictions should give ~25% pixel accuracy (4 classes)
preds = logits.argmax(dim=1)  # [2, 128, 128]
masks_sample = masks[:2].to(device)
pa = pixel_accuracy(preds, masks_sample)
print(f'Random-init pixel accuracy : {pa:.3f}  (expected ~0.25 for 4 classes)')

# 4. Loss computable
loss = F.cross_entropy(logits, masks_sample)
print(f'Initial cross-entropy loss : {loss.item():.4f}  (expected ~log(4) = {torch.log(torch.tensor(4.0)):.4f})')

print('All sanity checks passed!')

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

history = {'train_loss': [], 'val_loss': [], 'pixel_accuracy': [], 'mean_iou': [], 'dice': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, n_batches = 0.0, 0
    for batch_imgs, batch_masks in train_loader:
        batch_imgs  = batch_imgs.to(device)   # [B, 3, H, W]
        batch_masks = batch_masks.to(device)  # [B, H, W]
        optimizer.zero_grad()
        logits = model(batch_imgs)             # [B, 4, H, W]
        loss = F.cross_entropy(logits, batch_masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
        n_batches += 1
    history['train_loss'].append(t_loss / n_batches)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_pa, v_miou, v_dice, v_n = 0.0, 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for batch_imgs, batch_masks in val_loader:
            batch_imgs  = batch_imgs.to(device)
            batch_masks = batch_masks.to(device)
            logits = model(batch_imgs)
            preds  = logits.argmax(dim=1)      # [B, H, W]
            v_loss += F.cross_entropy(logits, batch_masks).item()
            v_pa   += pixel_accuracy(preds, batch_masks)
            v_miou += mean_iou(preds, batch_masks, num_classes=NUM_CLASSES)
            v_dice += dice_score(preds, batch_masks, num_classes=NUM_CLASSES)
            v_n += 1
    history['val_loss'].append(v_loss / v_n)
    history['pixel_accuracy'].append(v_pa / v_n)
    history['mean_iou'].append(v_miou / v_n)
    history['dice'].append(v_dice / v_n)
    scheduler.step()

    print(f'Epoch {epoch}/{NUM_EPOCHS}  '
          f'train={history["train_loss"][-1]:.4f}  '
          f'val={history["val_loss"][-1]:.4f}  '
          f'pxacc={history["pixel_accuracy"][-1]:.4f}  '
          f'mIoU={history["mean_iou"][-1]:.4f}  '
          f'dice={history["dice"][-1]:.4f}')

print('Training complete!')

## Evaluation

In [ ]:
# Full validation set evaluation
model.eval()
total_pa, total_miou, total_dice, n = 0.0, 0.0, 0.0, 0
with torch.no_grad():
    for batch_imgs, batch_masks in val_loader:
        batch_imgs  = batch_imgs.to(device)
        batch_masks = batch_masks.to(device)
        preds = model(batch_imgs).argmax(dim=1)
        total_pa   += pixel_accuracy(preds, batch_masks)
        total_miou += mean_iou(preds, batch_masks, num_classes=NUM_CLASSES)
        total_dice += dice_score(preds, batch_masks, num_classes=NUM_CLASSES)
        n += 1

final_pa   = total_pa   / n
final_miou = total_miou / n
final_dice = total_dice / n

print(f'Final Pixel Accuracy : {final_pa:.4f}  ({final_pa*100:.1f}%)')
print(f'Final Mean IoU       : {final_miou:.4f}')
print(f'Final Dice Score     : {final_dice:.4f}')

metrics = {
    'pixel_accuracy': float(final_pa),
    'mean_iou': float(final_miou),
    'dice_score': float(final_dice),
    'final_train_loss': float(history['train_loss'][-1]),
    'final_val_loss': float(history['val_loss'][-1]),
    'num_params': n_params,
    'num_epochs': NUM_EPOCHS,
    'base_channels': BASE_CHANNELS,
}
save_metrics_json(metrics, run_dir / 'unet_metrics.json')
print(f'Metrics saved to: {run_dir}/unet_metrics.json')

## Visualization

In [ ]:
# Training curves
from src.visualization.plots import plot_training_curves
plot_training_curves(
    {'train_loss': history['train_loss'], 'val_loss': history['val_loss'],
     'mean_iou': history['mean_iou'], 'dice': history['dice']},
    save_path=run_dir / 'unet_training_curve.png',
    title='U-Net Training Curves (Synthetic Shapes Segmentation)',
)
print('Training curve saved.')

In [ ]:
# Get validation predictions for visualisation
model.eval()
vis_imgs, vis_masks = next(iter(val_loader))
with torch.no_grad():
    vis_preds = model(vis_imgs.to(device)).argmax(dim=1).cpu()  # [B, H, W]

# Overlay: predicted mask on first image
overlay_segmentation_mask(
    image=vis_imgs[0].numpy(),
    mask=vis_preds[0].numpy(),
    alpha=0.5,
    save_path=run_dir / 'unet_overlay.png',
    title='U-Net Predicted Segmentation Overlay',
    class_names=CLASS_NAMES,
)
print('Overlay saved.')

# Side-by-side: image | GT mask | predicted mask
plot_segmentation_examples(
    images=vis_imgs.numpy(),
    gt_masks=vis_masks.numpy(),
    pred_masks=vis_preds.numpy(),
    save_path=run_dir / 'unet_examples.png',
    title='U-Net: Image | GT Mask | Predicted Mask',
    n_examples=4,
)
print('Examples figure saved.')

In [ ]:
# Per-class IoU analysis
model.eval()
all_pred_list, all_mask_list = [], []
with torch.no_grad():
    for bi, bm in val_loader:
        all_pred_list.append(model(bi.to(device)).argmax(1).cpu())
        all_mask_list.append(bm)
all_preds_cat = torch.cat(all_pred_list)
all_masks_cat = torch.cat(all_mask_list)

from src.metrics.segmentation import intersection_over_union
print('Per-class IoU:')
for c, name in enumerate(CLASS_NAMES):
    iou_c = intersection_over_union(all_preds_cat, all_masks_cat, c)
    print(f'  {name:12s}: IoU = {iou_c:.4f}')

## Failure Cases

**Common U-Net failure patterns on synthetic shapes:**

1. **Class imbalance**: Background occupies ~70% of pixels. The model can achieve high pixel
   accuracy by simply predicting background everywhere. Mean IoU is more informative.

2. **Boundary blurriness**: At shape edges, upsampling with bilinear interpolation introduces
   slight blurring. The predicted mask may be 1–2 pixels off on boundaries.

3. **Overlapping shapes**: When two shapes overlap, the last-drawn shape overwrites the mask.
   The model sees the mixed colour but the mask only records the top shape — ambiguous training signal.

4. **Small shapes**: If `min_size` is small, some shapes may be only 10–20 pixels. After
   3 downsampling steps (128→16), small shapes may vanish from the bottleneck features.

5. **Colour leakage**: The dataset uses fixed colours per class. The model may learn colour
   shortcuts rather than shape geometry. Test by changing `SHAPE_COLORS` at inference time.

## Exercises

1. **Weighted cross-entropy**: Apply class weights inversely proportional to class frequency
   (use `torch.bincount` on the mask). Does mean IoU improve for small objects?

2. **Deeper U-Net**: Add a 4th DownBlock (128→256) and corresponding UpBlock. How does this
   affect both parameter count and mean IoU?

3. **Skip connection ablation**: Remove all skip connections (set `UpBlock` inputs to only
   the upsampled decoder features). How much does mean IoU drop?

4. **Feature map visualisation**: After the bottleneck, reshape the [B, 256, 16, 16] feature
   map and plot the first 16 channels as a 4×4 grid. What spatial patterns do they encode?

5. **Boundary loss**: Implement a boundary-aware loss that penalises errors at shape edges more
   heavily. Use `torch.gradient` or morphological erosion to find boundary pixels.

## Key Takeaways

- **Semantic segmentation** is pixel-wise classification — the output has the same spatial size as input.
- **Encoder** builds rich semantic features by progressively downsampling.
- **Skip connections** restore spatial precision in the decoder — without them, boundaries would be blurry.
- **DoubleConv** blocks are more parameter-efficient than single large-kernel convolutions.
- **Pixel accuracy** alone is misleading for imbalanced datasets — always report mean IoU and Dice.
- U-Net has become the standard backbone for medical image segmentation due to its excellent
  precision-efficiency trade-off.
- Next notebook (12) replaces the CNN encoder with a Vision Transformer.

## Saved Outputs

In [ ]:
save_markdown_report(
    title='U-Net Semantic Segmentation',
    summary=(
        f'UNet (base_channels={BASE_CHANNELS}) trained for {NUM_EPOCHS} epochs on SyntheticShapes. '
        f'Pixel acc: {final_pa:.4f}, mean IoU: {final_miou:.4f}, Dice: {final_dice:.4f}.'
    ),
    metrics=metrics,
    figures=['unet_training_curve.png', 'unet_overlay.png', 'unet_examples.png'],
    tables=[],
    output_path=run_dir / 'unet_report.md',
    device=str(device),
    hyperparams={
        'base_channels': BASE_CHANNELS,
        'image_size': IMAGE_SIZE,
        'num_classes': NUM_CLASSES,
        'batch_size': BATCH_SIZE,
        'epochs': NUM_EPOCHS,
        'lr': LR,
    },
    architecture='DownBlock×3 + Bottleneck + UpBlock×3 (skip connections) + Conv1x1',
    loss_fn='CrossEntropyLoss (pixel-wise)',
)

update_runs_summary(
    session_name='unet_segmentation',
    report_path=run_dir / 'unet_report.md',
    metrics=metrics,
    figures=['unet_training_curve.png', 'unet_examples.png'],
)

print('All outputs saved:')
print(f'  {run_dir}/unet_metrics.json')
print(f'  {run_dir}/unet_training_curve.png')
print(f'  {run_dir}/unet_overlay.png')
print(f'  {run_dir}/unet_examples.png')
print(f'  {run_dir}/unet_report.md')